# HAALSI data cleaning, merging and audit notebook

This notebook documents the supplied 404-participant analytical file and audits it against the four raw archives. It preserves the historical cleaned CSV. It does **not** claim that every historical transformation has been exactly reconstructed: the original preprocessing script was not included, and some feature mappings (notably `quality`, `insufficiency`, and several covariates) still require confirmation.

**Run from the project root** with the package folders visible. The notebook writes audit reports under `data/audit/`; it does not overwrite `data/processed/haalsi_404_cleaned.csv`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import io
import zipfile
import pandas as pd
import numpy as np

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
AUDIT = ROOT / "data" / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)


## 2. Load the frozen 404-participant dataset

The participant cohort is frozen for comparison. `haalsi_original_reference.csv` is an integrity reference, not a raw source file.

In [ ]:
cleaned = pd.read_csv(PROCESSED / "haalsi_404_cleaned.csv")
reference = pd.read_csv(PROCESSED / "haalsi_original_reference.csv")

print("Cleaned dataset:", cleaned.shape)
print("Reference dataset:", reference.shape)
print("Unique participant IDs:", cleaned["prim_key"].nunique())
print("Duplicate IDs:", cleaned["prim_key"].duplicated().sum())
assert cleaned.shape == (404, 60)
assert cleaned["prim_key"].is_unique
assert reference["prim_key"].is_unique


## 3. Inspect the raw archives

The Wave 2 survey ZIP in this package contains a Stata file named `Longitudinal Public Release Dataset V5.dta`. This differs from the earlier source register entry naming V4 and must be reconciled with the original download record.

In [ ]:
ARCHIVES = {
    "dementia_w1": ("HAALSI Dementia Wave 1.zip", "HAALSI_DEMENTIA_Respondent-1.dta"),
    "dementia_w2": ("HAALSI Dementia Wave 2.zip", "HAALSI_DEMENTIA_Respondent_W2.dta"),
    "survey_w2": ("HAALSI Wave 2 Survey.zip", "Longitudinal Public Release Dataset V5.dta"),
    "survey_w1_w3": ("HAALSI Wave 3 Survey.zip", "HAALSI W1-3 Longitudinal Dataset.dta"),
}

def read_stata_from_zip(zip_name, member_suffix):
    with zipfile.ZipFile(RAW / zip_name) as z:
        matches = [n for n in z.namelist() if n.endswith(member_suffix)]
        assert len(matches) == 1, (zip_name, matches)
        return pd.read_stata(io.BytesIO(z.read(matches[0])), convert_categoricals=False)

dementia_w1 = read_stata_from_zip(*ARCHIVES["dementia_w1"])
dementia_w2 = read_stata_from_zip(*ARCHIVES["dementia_w2"])
survey_w2 = read_stata_from_zip(*ARCHIVES["survey_w2"])
survey_w1_w3 = read_stata_from_zip(*ARCHIVES["survey_w1_w3"])

for name, frame in [("Dementia W1", dementia_w1), ("Dementia W2", dementia_w2),
                    ("Survey W2 release", survey_w2), ("Combined W1-W3 survey", survey_w1_w3)]:
    print(name, frame.shape, "unique IDs:", frame["prim_key"].nunique())


## 4. Check participant linkage and coverage

All joins are performed on `prim_key`. Before merging, check uniqueness to avoid accidental many-to-many row multiplication. This section reports coverage; it does not infer why a participant was included or excluded historically.

In [ ]:
cohort_ids = set(cleaned["prim_key"].astype(str))
source_frames = {"dementia_w1": dementia_w1, "dementia_w2": dementia_w2,
                 "survey_w2": survey_w2, "survey_w1_w3": survey_w1_w3}
coverage = []
for name, frame in source_frames.items():
    assert frame["prim_key"].is_unique, f"Duplicate prim_key in {name}"
    ids = set(frame["prim_key"].dropna().astype(str))
    coverage.append({"source": name, "rows": len(frame), "unique_ids": len(ids),
                     "cohort_ids_found": len(cohort_ids & ids),
                     "cohort_ids_missing": len(cohort_ids - ids)})
coverage_df = pd.DataFrame(coverage)
coverage_df.to_csv(AUDIT / "cohort_id_coverage_notebook.csv", index=False)
coverage_df


## 5. Verify recorded MMSE totals

The dementia respondent files label `mmse_total` and `w2_mmse_total` as Mini Mental total scores. Compare these direct source values with `mmse_w1` and `mmse_w2`. Keep assessment status separate: a recorded score does not prove that the full assessment was completed.

In [ ]:
mmse_check = cleaned[["prim_key", "mmse_w1", "mmse_w2"]].merge(
    dementia_w1[["prim_key", "mmse_total", "mmse_status"]],
    on="prim_key", how="left", validate="one_to_one"
).merge(
    dementia_w2[["prim_key", "w2_mmse_total", "w2_mmse_status"]],
    on="prim_key", how="left", validate="one_to_one"
)
mmse_check["w1_matches"] = mmse_check["mmse_w1"].eq(mmse_check["mmse_total"]) | (mmse_check["mmse_w1"].isna() & mmse_check["mmse_total"].isna())
mmse_check["w2_matches"] = mmse_check["mmse_w2"].eq(mmse_check["w2_mmse_total"]) | (mmse_check["mmse_w2"].isna() & mmse_check["w2_mmse_total"].isna())
mmse_check.to_csv(AUDIT / "mmse_source_comparison_notebook.csv", index=False)
mmse_check[["w1_matches", "w2_matches"]].mean()


## 6. Review sleep source variables

The combined W1–W3 survey file contains sleep fields with labels shown below. `sp003` is reported hours of sleep; `sp001` and `sp002` provide bedtime and wake-time hour/minute; `sp004` asks whether the respondent got enough sleep; `sp005` asks how often the respondent wakes during the night. The cleaned CSV uses `quality_*` and `insufficiency_*` names, but their exact historical source mappings and recoding are not established by the available original cleaning code. Do not rename or reverse-code them until the codebook and value labels are checked.

In [ ]:
sleep_vars = [f"{w}sp{k}" for w in ("w1", "w2", "w3") for k in ("001_hour", "001_minute", "002_hour", "002_minute", "003", "004", "005")]
[(v, v in survey_w1_w3.columns) for v in sleep_vars]


## 7. Demonstrate candidate time and duration cleaning rules

These are candidate rules recorded in the earlier reconstruction notes. They are useful for an independent audit, but they must not be assumed to be the exact historical transformations unless the comparison reports support that conclusion.

In [ ]:
def valid_decimal_time(hour, minute):
    hour = pd.to_numeric(hour, errors="coerce")
    minute = pd.to_numeric(minute, errors="coerce")
    valid = hour.between(0, 23) & minute.between(0, 59)
    return (hour + minute / 60).where(valid)

def valid_duration(raw_duration):
    value = pd.to_numeric(raw_duration, errors="coerce")
    # 0-24 is a broad instrument-range check, not a clinical plausibility threshold.
    return value.where(value.between(0, 24))

# Example for each wave; retain missing values and do not overwrite the cleaned CSV.
sleep_examples = {}
for w in ("w1", "w2", "w3"):
    sleep_examples[f"{w}_bedtime_candidate"] = valid_decimal_time(
        survey_w1_w3[f"{w}sp001_hour"], survey_w1_w3[f"{w}sp001_minute"]
    )
    sleep_examples[f"{w}_wake_candidate"] = valid_decimal_time(
        survey_w1_w3[f"{w}sp002_hour"], survey_w1_w3[f"{w}sp002_minute"]
    )
    sleep_examples[f"{w}_duration_candidate"] = valid_duration(survey_w1_w3[f"{w}sp003"])

# Analyst-defined 3-16 hour flag is separate from the broad 0-24 validity check.
for w in ("w1", "w2", "w3"):
    sleep_examples[f"{w}_duration_3to16_flag"] = sleep_examples[f"{w}_duration_candidate"].between(3, 16)


## 8. Derived feature definitions and unresolved points

The final table contains across-wave means, changes, variability and a shifted bedtime summary. The exact original formulas and missing-value handling are not recoverable from the supplied minimal notebook. Before claiming a raw-to-clean reproduction, verify: (1) which waves are used in each change, (2) sample versus population standard deviation, (3) whether means use available waves or require complete waves, (4) the bedtime midnight-wrap rule, and (5) the source mapping and direction of the `quality` and `insufficiency` fields. The 404-row dataset is preserved until these checks pass.

## 9. Run the reusable audit script

From the package root, run `python scripts/00_audit_raw_sources.py`. It writes source inventory, cohort coverage, MMSE comparisons and sleep candidate comparisons to `data/audit/`. It does not overwrite the analytical CSV.

In [ ]:
# Optional: inspect the reports after running scripts/00_audit_raw_sources.py
for report in sorted(AUDIT.glob("*.csv")):
    print(report.name, pd.read_csv(report).shape)


## 10. Final integrity checks

In [ ]:
assert cleaned.shape == (404, 60)
assert cleaned["prim_key"].is_unique
assert reference["prim_key"].is_unique
assert cleaned["prim_key"].tolist() == reference["prim_key"].tolist()
print("Frozen historical analytical dataset preserved; audit complete.")
